# TripMe Part 11 — Corrective Adapter Re-evaluation
Attach `tripme-part11-corrective-eval`, enable a T4 GPU, keep Internet on, and Run All. The references remain provisional AI drafts and are not human-approved gold labels.

In [ ]:
!pip install -q --no-cache-dir transformers==4.48.3 accelerate==1.3.0 peft==0.14.0 bitsandbytes==0.48.2 safetensors>=0.4
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import torch, bitsandbytes as bnb
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'bitsandbytes:', bnb.__version__)
assert torch.cuda.is_available(), 'GPU is not enabled'
assert torch.cuda.device_count() == 1

In [ ]:
from pathlib import Path
import json, re, shutil, statistics, time
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
MAX_NEW_TOKENS = 384
matches = list(Path('/kaggle/input').rglob('provisional_gold_eval_120.jsonl'))
if not matches:
    raise FileNotFoundError('Attach tripme-part11-corrective-eval')
DATA_DIR = matches[0].parent
ADAPTER_DIR = DATA_DIR/'adapter'
OUTPUT_DIR = Path('/kaggle/working/tripme-part11-eval')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rows = [json.loads(line) for line in matches[0].read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(rows) == 120
assert (ADAPTER_DIR/'adapter_model.safetensors').is_file()
print('GPU:', torch.cuda.get_device_name(0), 'Rows:', len(rows))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant, device_map={'': 0}, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()
print('Corrective adapter loaded')

In [ ]:
def normalized(text):
    return re.sub(r'[^\w\u0D80-\u0DFF\u0B80-\u0BFF]+', ' ', text.casefold()).strip()

def repetition_score(text):
    words = normalized(text).split()
    if len(words) < 6: return 0.0
    trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
    return round(1-len(set(trigrams))/len(trigrams), 4)

def scenario_relevance(row, answer):
    lower = answer.casefold()
    groups = {
      'price': ['මිල','ගාස්තුව','වියදම්','price','fee','cost'],
      'uncertain': ['තහවුරු කළ නොහැක','සඳහන් නොවේ','අනුමාන','confirm karanna ba','data wala naha','cannot confirm','does not include','official'],
      'access': ['ප්‍රවේශ','පහසුකම්','රෝද පුටුව','accessib','wheelchair','entrance'],
      'etiquette': ['උපදෙස්','නීති','ගෞරව','instructions','rules','respect'],
    }
    hit = lambda key: any(term in lower for term in groups[key])
    scenario = row['scenario']
    if scenario in {'budget','uncertainty_current'}:
        return hit('price') and hit('uncertain') and not (hit('access') and not hit('price'))
    if scenario == 'family_accessibility': return hit('access') and hit('uncertain')
    if scenario == 'culture_etiquette': return hit('etiquette')
    return True

def score(row, answer, token_count):
    norm = normalized(answer)
    names = [normalized(name).rstrip('.') for name in row['target_place_names']]
    return {
      'nonempty': bool(answer),
      'complete_ending': bool(answer) and answer.rstrip().endswith(('.', '!', '?', '।')),
      'hit_token_limit': token_count >= MAX_NEW_TOKENS,
      'place_name_coverage': round(sum(name in norm for name in names)/max(1,len(names)),4),
      'scenario_relevance': scenario_relevance(row, answer),
      'repetition_score': repetition_score(answer),
      'encoding_clean': '�' not in answer and 'ā' not in answer,
    }

def generate(row):
    messages=[{'role':'system','content':row['system_prompt']},{'role':'user','content':row['prompt']}]
    inputs=tokenizer.apply_chat_template(messages,add_generation_prompt=True,return_tensors='pt').to(model.device)
    start=time.time()
    with torch.no_grad():
        output=model.generate(inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,repetition_penalty=1.08,pad_token_id=tokenizer.eos_token_id,eos_token_id=tokenizer.eos_token_id)
    new_tokens=output[0][inputs.shape[-1]:]
    return tokenizer.decode(new_tokens,skip_special_tokens=True).strip(),int(new_tokens.shape[-1]),round(time.time()-start,3)

In [ ]:
results=[]
for index,row in enumerate(rows,1):
    answer,tokens,seconds=generate(row)
    checks=score(row,answer,tokens)
    results.append({
      'eval_id':row['eval_id'],'language':row['language'],'scenario':row['scenario'],
      'target_place_ids':row['target_place_ids'],'target_place_names':row['target_place_names'],
      'prompt':row['prompt'],'reference_answer':row['reference_answer'],
      'generated':answer,'generated_tokens':tokens,'generation_seconds':seconds,
      'automatic_checks':checks,'evaluation_status':'provisional_ai_draft_not_human_gold',
    })
    if index%10==0: print('Completed:',index,'/',len(rows))

In [ ]:
def aggregate(items):
    checks=[row['automatic_checks'] for row in items]
    return {
      'rows':len(items),'nonempty_rate':sum(x['nonempty'] for x in checks)/len(checks),
      'complete_ending_rate':sum(x['complete_ending'] for x in checks)/len(checks),
      'token_limit_hit_rate':sum(x['hit_token_limit'] for x in checks)/len(checks),
      'mean_place_name_coverage':statistics.mean(x['place_name_coverage'] for x in checks),
      'scenario_relevance_rate':sum(x['scenario_relevance'] for x in checks)/len(checks),
      'encoding_clean_rate':sum(x['encoding_clean'] for x in checks)/len(checks),
      'mean_repetition_score':statistics.mean(x['repetition_score'] for x in checks),
      'mean_generation_seconds':statistics.mean(row['generation_seconds'] for row in items),
    }
summary={
 'status':'part11_corrective_evaluation_complete_not_human_gold',
 'model_id':MODEL_ID,'max_new_tokens':MAX_NEW_TOKENS,'all_120':aggregate(results),
 'by_language':{lang:aggregate([row for row in results if row['language']==lang]) for lang in ['si','singlish','en']},
 'warning':'Automatic checks are diagnostic; human review is required before release.',
}
with (OUTPUT_DIR/'corrective_generations.jsonl').open('w',encoding='utf-8') as handle:
    for row in results: handle.write(json.dumps(row,ensure_ascii=False)+'\n')
(OUTPUT_DIR/'evaluation_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
archive=shutil.make_archive('/kaggle/working/tripme_part11_eval_output','zip',OUTPUT_DIR)
print(json.dumps(summary,ensure_ascii=False,indent=2)); print('Download:',archive)

In [ ]:
from IPython.display import HTML,display
zip_path=Path('/kaggle/working/tripme_part11_eval_output.zip')
assert zip_path.is_file()
print('Exists:',zip_path.exists(),'Size MB:',round(zip_path.stat().st_size/1024/1024,2))
display(HTML("<a href='files/tripme_part11_eval_output.zip' download>Download tripme_part11_eval_output.zip</a>"))